#### Import des librairies

In [1]:
import pandas as pd
from tqdm import tqdm
import numpy as np

In [2]:

ENTREPOT_PATH = '~/Bureau/utils/data/'
DIRODUR_FILES_PATH = '~/Bureau/projets/DIRODUR/magasin/'
df = {}

#### Import des données

In [ ]:
# ----------------------------- #
# IMPORT DES DONNÉES DATAGROSYST#
# ----------------------------- #


def import_df(df_name, path_data, sep, index_col=None):
    df[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, index_col=index_col, low_memory=False).replace({'\r\n': '\n'}, regex=True)

def import_dfs(df_names, path_data, sep = ',', index_col=None, verbose=False):
    for df_name in tqdm(df_names) : 
        if(verbose) :
            print(" - ", df_name)
        import_df(df_name, path_data, sep, index_col=index_col)

# - noeuds_synthetise
# - connection_synthetise
# - noeuds_synthetise_restructure
# - itk_synthetise_agrege
# - typologie_can_culture
# - poids_connexions_synthetise_rotation

tables_with_id = [
    'noeuds_synthetise',
    'connection_synthetise',
    'plantation_perenne_phases_synthetise',
    'plantation_perenne_synthetise', 
    'sdc', 
    'domaine', 
    'synthetise',

]

tables_without_id = [
    'typologie_can_culture',
    'plantation_perenne_synthetise_restructure',
    'noeuds_synthetise_restructure', 
    'poids_connexions_synthetise_rotation',
    'itk_synthetise_agrege', 
    'connection_synthetise_restructure',
]

# import des données de l'entrepôt avec la colonne 'id' en index 
import_dfs(tables_with_id, ENTREPOT_PATH, sep = ',', index_col='id', verbose=False)

# import des données du magasin
import_dfs(tables_without_id, ENTREPOT_PATH, sep = ',', verbose=False)

100%|██████████| 6/6 [00:03<00:00,  1.82it/s]


In [243]:
studied_ids = [
    'fr.inra.agrosyst.api.entities.practiced.PracticedSystem_32154e3e-fb7e-4d46-91ee-1ca1aee550f4',
    'fr.inra.agrosyst.api.entities.practiced.PracticedSystem_0b9df2c4-35bc-43c9-a084-d07ca4176350',
    'fr.inra.agrosyst.api.entities.practiced.PracticedSystem_db359092-4686-4702-a23f-d45bccc6c25a',
    'fr.inra.agrosyst.api.entities.practiced.PracticedSystem_d84497de-8a9c-46a6-83db-6ac429cf5a89',
    'fr.inra.agrosyst.api.entities.practiced.PracticedSystem_178e4aaa-498b-4276-8990-dfcf41949b48',
    'fr.inra.agrosyst.api.entities.practiced.PracticedSystem_5f59aede-14b1-480b-bf7b-135cad00d9bd',
    'fr.inra.agrosyst.api.entities.practiced.PracticedSystem_0812743d-ef0f-4414-aa35-a6aeb55fa382'
]

In [244]:
df['itk_synthetise_agrege_test'] = df['itk_synthetise_agrege'].loc[
    df['itk_synthetise_agrege']['synthetise_id'].isin(studied_ids)
]
df['connection_synthetise_test'] = df['connection_synthetise'].loc[
    df['connection_synthetise'].index.isin(df['itk_synthetise_agrege_test']['itk_id'])
]
df['connection_synthetise_restructure_test'] = df['connection_synthetise_restructure'].loc[
    df['connection_synthetise_restructure'].index.isin(df['itk_synthetise_agrege_test']['itk_id'])
]
df['noeuds_synthetise_test'] = df['noeuds_synthetise'].loc[
    df['noeuds_synthetise'].index.isin(df['connection_synthetise_test']['cible_noeuds_synthetise_id'])
]
df['noeuds_synthetise_restructure_test'] = df['noeuds_synthetise_restructure'].loc[
    df['noeuds_synthetise_restructure']['id'].isin(df['noeuds_synthetise_test'].index)
]

df['poids_connexions_synthetise_rotation_test'] = df['poids_connexions_synthetise_rotation'].loc[
    df['poids_connexions_synthetise_rotation']['connexion_id'].isin(df['connection_synthetise_test'].index)
]

df['plantation_perenne_phases_synthetise_test'] = df['plantation_perenne_phases_synthetise'].loc[
    df['plantation_perenne_phases_synthetise'].index.isin(df['itk_synthetise_agrege_test']['itk_id'])
]

df['sdc_test'] = df['sdc'].loc[
    df['sdc'].index.isin(df['itk_synthetise_agrege_test']['sdc_id'])
]
df['domaine_test'] = df['domaine'].loc[
    df['domaine'].index.isin(df['itk_synthetise_agrege_test']['domaine_id'])
]

df['plantation_perenne_synthetise_test'] = df['plantation_perenne_synthetise'].loc[
    df['plantation_perenne_synthetise'].index.isin(df['itk_synthetise_agrege_test']['plantation_perenne_synthetise_id'])
]

df['plantation_perenne_synthetise_restructure_test'] = df['plantation_perenne_synthetise_restructure'].loc[
    df['plantation_perenne_synthetise_restructure']['id'].isin(df['itk_synthetise_agrege_test']['plantation_perenne_synthetise_id'])
]

df['typologie_can_culture_test'] = df['typologie_can_culture'].loc[
    df['typologie_can_culture']['culture_id'].isin(df['noeuds_synthetise_restructure_test'].culture_id) |
    df['typologie_can_culture']['culture_id'].isin(df['plantation_perenne_synthetise_restructure'].culture_id)
]

In [245]:
# export 
path='./'
df['connection_synthetise_test'].to_csv(path+'connection_synthetise.csv')
df['connection_synthetise_restructure_test'].to_csv(path+'connection_synthetise_restructure.csv')
df['noeuds_synthetise_test'].to_csv(path+'noeuds_synthetise.csv')
df['itk_synthetise_agrege_test'].to_csv(path+'itk_synthetise_agrege'+'.csv')
df['noeuds_synthetise_restructure_test'].to_csv(path+'noeuds_synthetise_restructure'+'.csv')
df['typologie_can_culture_test'].to_csv(path+'typologie_can_culture'+'.csv')
df['poids_connexions_synthetise_rotation_test'].to_csv(path+'poids_connexions_synthetise_rotation'+'.csv')
df['sdc_test'].to_csv(path+'sdc'+'.csv')
df['domaine_test'].to_csv(path+'domaine'+'.csv')
df['plantation_perenne_phases_synthetise_test'].to_csv(path+'plantation_perenne_phases_synthetise'+'.csv')
df['plantation_perenne_synthetise_test'].to_csv(path+'plantation_perenne_synthetise'+'.csv')
df['plantation_perenne_synthetise_restructure_test'].to_csv(path+'plantation_perenne_synthetise_restructure'+'.csv')